# Aarohan — Stage 2: Full-Dataset Exploratory Data Analysis

**Project:** Aarohan (Intelligent Demand Planning for Perishable Retail)  
**Dataset:** FreshRetailNet-50K (Dingdong Inc.)  
**Partitions:** `data/raw/train.parquet` (4.5M rows) & `data/raw/eval.parquet` (350k rows)  
**Environment Constraints:** DuckDB memory limit 512 MB, 2 execution threads.  

This notebook provides a reproducible, end-to-end exploratory data analysis across all 4.85 million records, calling modular, tested analytical functions from `aarohan.analysis`.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

# Ensure project root and src/ are in sys.path
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from aarohan.analysis.diagnostics import (
    calculate_stockout_severity_summary,
    calculate_zero_sales_diagnostics,
    compare_train_eval_populations,
)
from aarohan.analysis.profiling import (
    calculate_daily_demand_summary,
    calculate_series_profiles,
    calculate_weekday_patterns,
    define_demand_segments,
)
from aarohan.config import load_config
from aarohan.data.load import duckdb_cursor, resolve_dataset_path

data_cfg = load_config("data")
train_path = resolve_dataset_path(data_cfg, "raw_train_path")
eval_path = resolve_dataset_path(data_cfg, "raw_eval_path")
print(f"Project root: {project_root}")
print(f"Train path: {train_path}")
print(f"Eval path: {eval_path}")

## 1. Overall Demand Distribution

We compute daily aggregate and quantile metrics using memory-efficient DuckDB queries with projection pushdown.

In [ ]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    daily_df = calculate_daily_demand_summary(con, train_path.as_posix())

print(f"Aggregated {len(daily_df)} dates.")
print(f"Total demand index sum: {daily_df['total_demand'].sum():,.2f}")
print(f"Average daily demand mean: {daily_df['mean_demand'].mean():.4f}")
cols = [
    "dt",
    "weekday_name",
    "total_demand",
    "mean_demand",
    "median_demand",
    "zero_sales_pct",
    "mean_daytime_stockout_pct",
]
daily_df[cols].head(10)

## 2. Series-Level Heterogeneity & Demand Segmentation

We summarize all 50,000 store-product series over the 90 training dates, evaluating mean demand, volatility (CV), intermittency, and stockout exposure.

In [ ]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    series_df = calculate_series_profiles(con, train_path.as_posix())
    series_df, thresholds_df = define_demand_segments(series_df)

print("--- Segmentation Boundaries & Profile Summary ---")
thresholds_df

## 3. Stockout Severity and Availability Diagnostics

We compare observed sales performance and zero-sales frequency across daytime-stockout-free, partially affected, and fully daytime-stockout days.

In [ ]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    severity_df = calculate_stockout_severity_summary(con, train_path.as_posix())
    zero_diag = calculate_zero_sales_diagnostics(con, train_path.as_posix())

print("--- Demand by Stockout Severity Category ---")
print(severity_df.to_string())
print("\n--- Zero-Sales Decomposition ---")
for k, v in zero_diag.items():
    print(f"{k}: {v}")

## 4. Calendar Patterns: Weekday vs Weekend

Analysis of day-of-week demand and stockout patterns demonstrates substantial weekend lift.

In [ ]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    weekday_df = calculate_weekday_patterns(con, train_path.as_posix())

weekday_df[
    [
        "weekday_name",
        "is_weekend",
        "date_count",
        "total_demand",
        "mean_demand",
        "median_demand",
        "mean_daytime_stockout_pct",
    ]
]

## 5. Training vs Held-Out Evaluation Population Comparison

> **Notice**: All evaluation analysis is strictly descriptive and held out. It verifies population alignment and is not used for model tuning.

In [ ]:
with duckdb_cursor(threads=2, max_memory="512MB") as con:
    comp_df = compare_train_eval_populations(
        con, train_path.as_posix(), eval_path.as_posix()
    )

comp_df

## 6. Generated Figures

All 8 publication-grade vector SVG figures generated by `aarohan.analysis.visualize` are saved in `reports/figures/`:
- `fig01_daily_aggregate_demand.svg`
- `fig02_series_mean_demand_distribution.svg`
- `fig03_zero_sales_distribution.svg`
- `fig04_weekday_demand_pattern.svg`
- `fig05_stockout_exposure_distribution.svg`
- `fig06_sales_by_stockout_category.svg`
- `fig07_demand_segments_stockout_profile.svg`
- `fig08_category_heterogeneity.svg`

Complete analytical commentary and guidance for Stage 3 are documented in `reports/eda_findings.md`.